# SKY130 GDS to a connected, tagged 3D finite-element mesh

This notebook reads native GDSII with **gdstk**, resolves hierarchy and mask
geometry, constructs process-aware 3D material/source regions, and creates a
conformal tetrahedral mesh with **Gmsh**. The default input is the bundled
`sram_sp_cell`. Generic mode accepts a user-supplied GDS file; external PDK example
bundles are optional and are not included in this repository.

Set `GDS_INPUT` before starting the kernel to switch files, for example:

```bash
GDS_CASE=generic GDS_INPUT=/absolute/path/to/custom_cell.gds \
  GDS_MESH_SIZE_UM=0.40 GDS_FINE_MESH_SIZE_UM=0.10 jupyter lab
```

Relative `GDS_INPUT` paths are resolved below this repository's `data/`;
absolute paths are also accepted for user-supplied layouts.  Unrelated GDS/GDS2 files still use
the general extraction path; the physical SKY130 stack is activated only
when its layer signature is detected (or when `GDS_LAYER_PROFILE=sky130`).

The geometry, meshing, validation, and SVG helpers are shared with Kelvin in
`mesh/gds_notebook.py`; the notebook does not maintain a separate mesher.

The staged workflow now defaults to the bundled **single SRAM bitcell**. Its shared polygon backend is also used by Kelvin's steady/transient pipeline. The final section imports the exact saved mesh without remeshing. Set `GDS_CASE=generic` and `GDS_INPUT` for a custom layout.


## SRAM bitcell: compare the old and polygon-preserving meshes

The standalone cell below visualizes `sram_sp_cell` from
`data/sram22_64x22m4w22.gds` (1.20 × 1.58 µm). It does not change
the assembled thermal workflow in Sections 1–8. This optional historical
comparison can be run independently of those sections.

The figures use the supplied reference's **two-panel SVG format**:
**original GDS on the left, 3D isometric geometry on the right**, with
separate old/new figures. Colors match the reference: LI1 red, M1 gold,
M2 blue, poly lavender, diffusion green, and contacts charcoal.

- **GDS:** all 26 polygon layer/datatype pairs, not just three layers.
- **Old:** all modeled device/interconnect layers from the existing
  tagged mesh. Bounding rectangles add material outside the GDS polygons.
- **New:** exact-polygon, separately meshed layer prisms made with this
  notebook's extraction and extrusion functions. These are **visualization
  previews, not a connected or solver-ready thermal mesh**.

The earlier three-layer view selected diffusion, LI1, and M1 only to
expose bounding-box errors; it was not the complete layer stack. Poly,
contacts, vias, M2, and nwell context are now included where modeled.
Implant, pin, mask-processing, and boundary layers remain visible in
the GDS but are not invented as independent stacked material films.
L64/D44 remains explicitly unmapped under the pinned layer table.

**Why LI1 looked merged:** eight raw LI1 polygons legitimately union into
six disconnected pieces because two rail extensions overlap their rails.
The new prism mesh retains all six. The old bounding-box thermal mesh
instead has three, incorrectly joining four central features across
140 nm gaps. The supplied reference illustrates original GDS polygons,
not the saved old solver mesh; its separate pieces are therefore correct.
These counts describe connectivity within LI1, not whole-circuit nets.

At the time these previews were generated, full assembly rejected
**0.02975 µm² of licon outside modeled poly/diff**. The integrated flow
now validates whole-cut landing relationships and memory-core coverage
explicitly; see `SRAM_CONTACT_GEOMETRY.md`. These historical
previews still do not supply thermal predictions. Mesh densities
are not matched; compare footprints rather than triangle counts.
Licon is drawn in the GDS and old geometry but has no guessed extrusion
in the new preview. Deep substrate and fill visibility/cut limits are
labeled on the figures; GDS layer count is not a count of material films.

Existing figures are displayed by default. Set `REBUILD_SRAM_COMPARISON`
to `True` to regenerate them. `KELVIN_COMPARE_PYTHON` can select a Python
environment with gdstk, Gmsh, NumPy, IPython, Matplotlib, and PyVista;
`KELVIN_OLD_MSH` can select an existing old mesh of the same bitcell.
Without an override, the current notebook interpreter is used; nothing is installed.
Regenerating this optional comparison also requires the historical external
`../data/sky130/pdk/` reference files used by the layer-audit helper. These are
not bundled; leave `REBUILD_SRAM_COMPARISON=False` in a fresh clone. The main
staged SRAM mesh/solver workflow below does not require those files.

[Comparison details and commands](out/sram_mesh_comparison/README.md) ·
[Complete layer inventory](out/sram_mesh_comparison/sram_all_layer_audit.md) ·
[LI1 gap diagnostic](out/sram_mesh_comparison/li1_connectivity.svg) ·
[Actual tetrahedral sections](out/sram_mesh_comparison/sram_notebook_mesh_sections.svg)

**Historical previews only:** their saved contact-failure message is not the status of the integrated pipeline below. The main staged workflow now builds the complete validated SRAM thermal mesh with resolved contact interpretation.


In [ ]:
# Standalone: no variables from the main thermal-meshing cells are required.
from pathlib import Path
import json
import os
import subprocess
import sys
from IPython.display import SVG, Markdown, display

REBUILD_SRAM_COMPARISON = False
comparison_start = Path.cwd().resolve()
comparison_kelvin = next((root
    for p in (comparison_start, *comparison_start.parents)
    for root in (p, p / 'kelvin')
    if (root / 'mesh/gds_notebook.py').is_file()
    and (root / 'read_gds.ipynb').is_file() and (root / 'data').is_dir()), None)
if comparison_kelvin is None:
    raise FileNotFoundError('Run this notebook from the Kelvin repository or its parent')
comparison_out = comparison_kelvin / 'out/sram_mesh_comparison'
comparison_old = Path(os.environ.get('KELVIN_OLD_MSH', str(
    comparison_kelvin / 'out/periodic_verification_dos7Z2/periodic/gds_volume.msh')))
if not comparison_old.is_absolute():
    comparison_old = (comparison_kelvin / comparison_old).resolve()
comparison_python = os.environ.get('KELVIN_COMPARE_PYTHON', sys.executable)
if REBUILD_SRAM_COMPARISON:
    if not comparison_old.is_file():
        raise FileNotFoundError('Set KELVIN_OLD_MSH to an existing old sram_sp_cell mesh')
    comparison_env = os.environ.copy()
    comparison_deps = comparison_kelvin / '.deps/dolfinx-mpc-0.10/python'
    comparison_env['PYTHONPATH'] = os.pathsep.join(filter(None, [
        str(comparison_kelvin), str(comparison_deps), comparison_env.get('PYTHONPATH', '')]))
    comparison_env['PYTHONDONTWRITEBYTECODE'] = '1'
    print('Regenerating using:', comparison_python)
    subprocess.run([comparison_python, 'cases/build_sram_polygon_preview.py'],
                   cwd=comparison_kelvin, env=comparison_env, check=True)
    subprocess.run([comparison_python, 'cases/audit_sram_layers.py'],
                   cwd=comparison_kelvin, env=comparison_env, check=True)
    subprocess.run([comparison_python, 'cases/compare_sram_meshes.py',
                    '--old-msh', str(comparison_old)],
                   cwd=comparison_kelvin, env=comparison_env, check=True)
    subprocess.run([comparison_python, 'cases/render_sram_notebook_comparison.py',
                    '--old-msh', str(comparison_old)],
                   cwd=comparison_kelvin, env=comparison_env, check=True)
    subprocess.run([comparison_python, 'cases/check_sram_li1.py',
                    '--old-msh', str(comparison_old)],
                   cwd=comparison_kelvin, env=comparison_env, check=True)
comparison_summary_path = comparison_out / 'new_exact_footprint_meshes.json'
comparison_figures = ('sram_reference_style_new.svg',
                      'sram_reference_style_old.svg')
if comparison_summary_path.is_file() and all((comparison_out / name).is_file() for name in comparison_figures):
    comparison_summary = json.loads(comparison_summary_path.read_text())
    display(Markdown('**Historical disconnected layer previews**, not the solver mesh below. '
                     + 'Saved validation status: ' + str(comparison_summary['new_thermal_flow_failure'])))
    for comparison_figure in comparison_figures:
        display(SVG(filename=str(comparison_out / comparison_figure)))
else:
    print('Optional comparison previews absent. Continue below for the solver mesh,')
    print('or set REBUILD_SRAM_COMPARISON = True to recreate the historical comparison.')


## 1. Configuration, provenance, and what the PDK does not contain

An optional externally obtained bundle in `data/sky130/` may contain
official example layouts and their PDK references. If its
`data/sky130/sources.json` is present, every listed artifact is checked.
The bundled SRAM needs no parent-project data or external example bundle.

The notebook deliberately separates:

- **PDK facts:** XY masks/purposes and documented vertical dimensions;
- **thermal assumptions:** substrate truncation and material-property
  mappings, which are not foundry-qualified by the public PDK;
- **power inputs:** workload-dependent W or W/m³ values, which GDS cannot
  provide.

The open_pdks/Magic technology file, public stack diagram, and routing TLEF
use slightly different abstractions (for example, Metal-1 is 0.36 µm in the
Magic height table and 0.35 µm in the TLEF).  This notebook uses the Magic
height entries, shifted so active top/poly bottom is z=0, and records the
discrepancy in the manifest.  This is a reproducible *working geometry*,
not a foundry-qualified 3D process deck.  A configurable finite substrate
depth is a model truncation, not wafer thickness.

**SRAM default:** `data/sram22_64x22m4w22.gds`, cell `sram_sp_cell`; output: `out/sram_notebook_mesh/`. The preset retains Kelvin's 50.3262 µm active-top-to-backside distance, 10 nm heat sources, SiO₂ field background and nominal 1 µm SiN cap. `GDS_REFINE` scales target lengths (smaller = finer). z=0 is active top. These are modeling assumptions, not a fully calibrated process stack.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from html import escape
from pathlib import Path
from collections import defaultdict
import colorsys
import csv
import hashlib
import json
import math
import os
import re

import gdstk
import numpy as np
from IPython.display import HTML, SVG, display


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        for root in (candidate, candidate / "kelvin"):
            if ((root / "mesh/gds_notebook.py").is_file()
                    and (root / "data").is_dir()
                    and (root / "read_gds.ipynb").is_file()):
                return root
    raise FileNotFoundError("Run this notebook from the Kelvin repository or its parent")


ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = ROOT / "data"
SKY130_BUNDLE_DIR = DATA_DIR / "sky130"
SKY130_PDK_DIR = SKY130_BUNDLE_DIR / "pdk"
SKY130_SOURCE_MANIFEST_PATH = SKY130_BUNDLE_DIR / "sources.json"

# A configuration rerun invalidates the old region-construction state.
# Rebuild regions before meshing, even if a previous cell left them in memory.
globals().pop("REGION_LAYOUT_IDENTITY", None)
# Reset SRAM-only optional defaults when switching back to generic mode.
CHANNEL_DEPTH_UM = None
ACTIVE_BACKGROUND_MATERIAL = "Si_bulk"
TOP_PASSIVATION_THICKNESS_UM = 0.0
TOP_PASSIVATION_MATERIAL = "SiN"

# Generic mode requires an explicit user-supplied GDS_INPUT.
SRAM_PIPELINE = os.environ.get("GDS_CASE", "sram").lower() == "sram"
INPUT_GDS = os.environ.get("GDS_INPUT", "sram22_64x22m4w22.gds" if SRAM_PIPELINE else "")
if not INPUT_GDS:
    raise ValueError("Generic mode requires GDS_INPUT (absolute or relative to repo/data)")
TOP_CELL_NAME = os.environ.get("GDS_TOP_CELL") or ("sram_sp_cell" if SRAM_PIPELINE else None)
MESH_SIZE_UM = float(os.environ.get("GDS_MESH_SIZE_UM", "0.40"))
FINE_MESH_SIZE_UM = float(os.environ.get("GDS_FINE_MESH_SIZE_UM", "0.10"))
REFINEMENT_DISTANCE_UM = float(os.environ.get("GDS_REFINEMENT_DISTANCE_UM", "0.35"))
MIN_TET_QUALITY = float(os.environ.get("GDS_MIN_TET_QUALITY", "0.025"))
SUBSTRATE_DEPTH_UM = float(os.environ.get("GDS_SUBSTRATE_DEPTH_UM", "2.0"))
PLACEHOLDER_LAYER_THICKNESS_UM = 1.0
MAX_FLATTENED_POLYGONS = 250_000
IGNORED_GDS_SPECS = set()
LAYER_MAP_PROFILE = os.environ.get("GDS_LAYER_PROFILE", "auto").strip().lower()
if LAYER_MAP_PROFILE not in {"auto", "none", "scmos", "sky130"}:
    raise ValueError("GDS_LAYER_PROFILE must be one of: auto, none, scmos, sky130")
if not (0 < FINE_MESH_SIZE_UM <= MESH_SIZE_UM):
    raise ValueError("GDS_FINE_MESH_SIZE_UM must be positive and <= GDS_MESH_SIZE_UM")
if SUBSTRATE_DEPTH_UM <= 0.3262:
    raise ValueError("substrate depth must exceed the SKY130 well bottom at 0.3262 um")

# User-supplied records override an inferred PDK profile.  They remain the
# general escape hatch for other processes.
PROCESS_STACK_RECORDS = []

SCMOS_LAYER_NAMES = {
    41: "p-well", 42: "n-well", 43: "active", 44: "p+ select",
    45: "n+ select", 46: "poly", 47: "poly contact",
    48: "active contact", 49: "metal 1", 50: "via 1", 51: "metal 2",
}
SCMOS_LAYER_COLORS = {
    41: "#C8A97E", 42: "#E6B566", 43: "#58A65C", 44: "#E88AB8",
    45: "#7CC7E8", 46: "#D85C4A", 47: "#585858", 48: "#7A7A7A",
    49: "#4D78C4", 50: "#2F2F2F", 51: "#B45AC9",
}
SCMOS_SIGNATURE = frozenset(SCMOS_LAYER_NAMES)
SCMOS_CONNECTOR_RULES = (
    (46, 47, 49, "poly → poly contact → metal 1"),
    (43, 48, 49, "active → active contact → metal 1"),
    (49, 50, 51, "metal 1 → via 1 → metal 2"),
)

# Authoritative GDS purposes from pdk/gds_layers.csv.
SKY130_SPECS = {
    "nwell": (64, 20), "diff": (65, 20), "poly": (66, 20),
    "licon1": (66, 44), "li1": (67, 20), "mcon": (67, 44),
    "met1": (68, 20), "via": (68, 44), "met2": (69, 20),
    "via2": (69, 44), "met3": (70, 20), "via3": (70, 44),
    "met4": (71, 20), "via4": (71, 44), "met5": (72, 20),
    "areaid_sc": (81, 4),
}
SKY130_SIGNATURE = frozenset({
    SKY130_SPECS[name]
    for name in ("nwell", "diff", "poly", "licon1", "li1", "mcon", "met1")
})
SKY130_LAYER_COLORS = {
    "nwell": "#8ECAE6", "diff": "#7fc08a", "poly": "#c9a0dc",
    "licon1": "#2b2b2b", "li1": "#d1342a", "mcon": "#2b2b2b",
    "met1": "#e0a840", "via": "#2b2b2b", "met2": "#3d6fd6",
    "via2": "#E11D48", "met3": "#0891B2", "via3": "#FB923C",
    "met4": "#C026D3", "via4": "#FBBF24", "met5": "#059669",
}

# Working geometry derived from the pinned open_pdks/Magic height entries,
# shifted by -0.3262 um so active top/poly bottom is z=0.  The process and
# liner detail does not map one-to-one to routing abstractions; this is a
# documented thermal solid approximation, not a foundry 3D process deck.
SKY130_SOLID_Z_UM = {
    "well": (-0.3262, -0.1200),
    "diff": (-0.1200, 0.0000),
    "poly": (0.0000, 0.1800),
    "li1": (0.6099, 0.7099),
    "mcon": (0.7099, 1.0499),
    "met1": (1.0499, 1.4099),
    "via": (1.4099, 1.6799),
    "met2": (1.6799, 2.0399),
    "via2": (2.0399, 2.4599),
    "met3": (2.4599, 3.3049),
    "via3": (3.3049, 3.6949),
    "met4": (3.6949, 4.5399),
    "via4": (4.5399, 5.0449),
    "met5": (5.0449, 6.3049),
}
SKY130_DIELECTRIC_CEILINGS_UM = (0.6099, 1.0499, 1.6799, 2.4599, 3.6949, 5.0449, 11.5572)

# One backend is used by this notebook and production SRAM simulations.
# Binding to globals keeps staged notebook configuration visible to helpers.
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from mesh.gds_notebook import install_helpers
install_helpers(globals())
if SRAM_PIPELINE:
    from mesh.sram import thermal_context_options
    # Preserve previous SRAM thermal inputs; smaller GDS_REFINE means finer.
    globals().update(thermal_context_options(float(os.environ.get("GDS_REFINE", "1.0"))))
    print("SRAM preset: 50.3262 um depth, 10 nm sources, field SiO2, 1 um SiN cap")

input_path = Path(INPUT_GDS).expanduser()
GDS_PATH = input_path.resolve() if input_path.is_absolute() else (DATA_DIR / input_path).resolve()
if not input_path.is_absolute():
    try:
        GDS_PATH.relative_to(DATA_DIR.resolve())
    except ValueError as exc:
        raise ValueError("Relative GDS_INPUT must remain below data/; use an absolute path instead") from exc
if GDS_PATH.suffix.lower() not in {".gds", ".gds2"}:
    raise ValueError("GDS_INPUT must name a .gds or .gds2 file")
if not GDS_PATH.is_file():
    raise FileNotFoundError(GDS_PATH)

sky130_sources = (
    json.loads(SKY130_SOURCE_MANIFEST_PATH.read_text(encoding="utf-8"))
    if SKY130_SOURCE_MANIFEST_PATH.is_file() else None
)


## 2. Read, flatten, and normalize mask geometry

The selected top cell is flattened recursively, so nested SREF/AREF transformations, rotations, reflections, magnification, repetitions, and paths are resolved by gdstk. Geometry is grouped by `(layer, datatype)` and unioned on the original GDS database grid. Any weakly-simple polygon encoding a hole is fractured into simple pieces suitable for OCC.

Boolean union intentionally discards per-polygon GDS properties; material identity is assigned later by the process stack.


In [ ]:
# Layout helpers are shared with Kelvin via mesh.gds_notebook.


## 3. Audit every layout, compare the SKY130 examples, and check contacts

Discovery is recursive below `data/`, so the pinned PDK examples and the
original GDS2 files use the same extraction code.  SHA-256 checks make the
downloaded bundle reproducible.  The comparison gallery shows the
fabrication masks used by the 3D builder; implants, pins, identifiers, and
boundaries remain visible in the tabular audit but never become fake
material slabs.

For SKY130, the notebook verifies both sides of every connector in XY:
`licon1` must be covered by `(diff OR poly)` and `li1`, and `mcon` must be
covered by `li1` and `met1`.  This confirms contact masks exist; their z
spans come from the process profile in the next section.


In [ ]:
verified_sky130_artifacts = verify_sky130_bundle(SKY130_BUNDLE_DIR, sky130_sources)
audited_layouts, audit_rows = discover_and_audit(ROOT / "data")
display(HTML(audit_html(audit_rows)))

AUDIT_PATH = ROOT / "out" / "gds_geometry" / "gds_audit.json"
AUDIT_PATH.parent.mkdir(parents=True, exist_ok=True)
AUDIT_PATH.write_text(
    json.dumps({
        "schema": "gds-audit-v2",
        "gdstk_version": gdstk.__version__,
        "verified_sky130_artifact_count": len(verified_sky130_artifacts),
        "files": audit_rows,
    }, indent=2) + "\n",
    encoding="utf-8",
)

try:
    input_key = GDS_PATH.relative_to(DATA_DIR).as_posix()
except ValueError:
    input_key = str(GDS_PATH)
if TOP_CELL_NAME is None and input_key in audited_layouts:
    layout = audited_layouts[input_key]
else:
    layout = extract_layout(GDS_PATH, TOP_CELL_NAME)

safe_stem, safe_top = safe_component(GDS_PATH.stem), safe_component(layout["top_cell"])
OUTPUT_DIR = (ROOT / "out" / "sram_notebook_mesh" if SRAM_PIPELINE else
              ROOT / "out" / "gds_geometry" / safe_stem / safe_top)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_STEM = "sram_sp_cell" if SRAM_PIPELINE else f"{safe_stem}_{safe_top}"

print(f"Selected: {input_key} / top {layout['top_cell']}")
print(f"Hierarchy: {layout['library_cell_count']} cells; dependencies={layout['dependency_cells']}")
print(
    f"Geometry: {layout['flattened_polygon_count']} flattened polygons -> "
    f"{layout['union_component_count']} union components across all mask purposes"
)
print(f"Layer/datatype pairs: {layout['observed_specs']}")
print(f"Bounding box: {layout['bbox_um']} um")

layer_profile = choose_layer_profile(layout, LAYER_MAP_PROFILE)
if layer_profile == "scmos":
    auto_ignored_specs = {
        spec for spec in layout["observed_specs"] if not 21 <= spec[0] <= 62
    }
    contact_audit = scmos_contact_audit(layout)
elif layer_profile == "sky130":
    sky130_semantic_specs = set(SKY130_SPECS.values()) | {(65, 44)}
    auto_ignored_specs = set(layout["observed_specs"]) - sky130_semantic_specs
    contact_audit = sky130_contact_audit(layout)
    # The shared audit validates each complete cut and explicitly accounts
    # for permitted memory-core overhangs; aggregate coverage is diagnostic.
else:
    auto_ignored_specs = set()
    contact_audit = []
display(HTML(contact_audit_html(layer_profile, contact_audit, auto_ignored_specs)))

sky130_example_keys = sorted(
    key for key in audited_layouts
    if key.startswith("sky130/gds/") and key.endswith(".gds")
)
if sky130_example_keys:
    cards = "".join(
        '<div style="flex:1;min-width:260px;border:1px solid #d9e2e8;border-radius:8px;padding:6px">'
        + sky130_gallery_svg(audited_layouts[key], Path(key).stem)
        + '</div>'
        for key in sky130_example_keys
    )
    display(HTML('<div style="display:flex;gap:10px;flex-wrap:wrap">' + cards + '</div>'))

power_sidecars = cell_power_sidecars(sky130_example_keys)
if power_sidecars:
    body = "".join(
        f"<tr><td><code>{escape(row['cell'])}</code></td><td>{row['transistor_count']}</td>"
        f"<td>{row['area_um2']:.4f}</td><td>{row['leakage_nw']:.6f}</td>"
        f"<td><code>{escape(row['spice'])}</code></td></tr>"
        for row in power_sidecars
    )
    display(HTML(
        '<h4>Matched electrical sidecars (TT, 25 °C, 1.8 V)</h4>'
        '<table><thead><tr><th>cell</th><th>transistors</th><th>area (µm²)</th>'
        '<th>Liberty cell leakage (nW)</th><th>SPICE view</th></tr></thead>'
        f'<tbody>{body}</tbody></table>'
        '<p style="font-size:12px;color:#546e7a">Leakage uses the common Liberty '
        '<code>1nW</code> unit. Dynamic power still requires slew, load, and activity.</p>'
    ))


## 4. Construct non-overlapping 3D material and source regions

For a recognized SKY130 layout, mask order is never treated as z order.
The builder uses the pinned process profile, splits `licon1` into contacts
landing on diffusion versus poly, partitions `poly ∩ diff` into NMOS and
PMOS candidate source regions (one tag per connected channel footprint),
and retains the remaining diffusion as doped source/drain silicon.  It then
fills every remaining part of each vertical slab with dielectric and
partitions the silicon well/active stack.

This is why separate electrical nets no longer appear as unexplained
thermally floating solids: conductors remain electrically distinct, while
dielectric and substrate provide the real thermal path between them.  The
finite substrate depth and thermal material choices are explicit modeling
assumptions.  Higher SKY130 metals are supported when their drawing masks
are present.

Other processes still accept explicit `PROCESS_STACK_RECORDS`.  With no
physical profile, the notebook retains its clearly marked visualization-only
mask extrusion rather than inventing foundry data.


In [ ]:
globals().pop("REGION_LAYOUT_IDENTITY", None)
explicit_user_stack = bool(PROCESS_STACK_RECORDS)
if layer_profile == "sky130" and not explicit_user_stack:
    regions, process_model, used_gds_specs = build_sky130_regions()
    ignored_specs = set(layout["observed_specs"]) - used_gds_specs
else:
    ignored_specs = {
        (int(layer), int(datatype)) for layer, datatype in IGNORED_GDS_SPECS
    } | auto_ignored_specs
    unknown_ignored = sorted(ignored_specs - set(layout["observed_specs"]))
    if unknown_ignored:
        raise ValueError(f"IGNORED_GDS_SPECS contains masks absent from the selected top: {unknown_ignored}")
    active_specs = [spec for spec in layout["observed_specs"] if spec not in ignored_specs]
    if not active_specs:
        raise ValueError("All observed masks were ignored; nothing remains to mesh")
    regions = generic_regions(active_specs)
    process_model = None
    used_gds_specs = set(active_specs)

if len({region["physical_tag"] for region in regions}) != len(regions):
    raise ValueError("Physical volume tags must be unique")
if len({region["name"] for region in regions}) != len(regions):
    raise ValueError("Physical region names must be unique")

print(f"3D regions: {len(regions)}; profile={layer_profile!r}")
for region in regions:
    volume = sum(v["area_um2"] for v in region["volumes"]) * (region["z_max_um"] - region["z_min_um"])
    print(
        f"  tag {region['physical_tag']:>2}  {region['name']:<30} "
        f"{region['role']:<10} z={region['z_min_um']:>7.4f}..{region['z_max_um']:<7.4f} um  "
        f"V={volume:.6f} um^3"
    )

# Bind this region build to the selected layout; partial case switches fail closed.
from mesh.gds_notebook import region_layout_identity
REGION_LAYOUT_IDENTITY = region_layout_identity(layout)


## 5. Validate topology and preview the solver-facing manifest

Every material/source region must be non-overlapping in positive volume.
For SKY130, each dielectric slice additionally satisfies
`conductor area + dielectric area = domain area` before extrusion.

This cell previews the current manifest in memory. It does not overwrite
a completed solver bundle. The mesh cell below snapshots the **current**
regions, material assignments and process model again, builds and validates
the mesh, and only then publishes the matching manifest. Rerunning the mesh
cell after editing a material therefore does not require rerunning this cell.


In [ ]:
from mesh.gds_notebook import build_region_manifest

manifest_preview = build_region_manifest(globals())
print(f"Validated current manifest preview: {len(manifest_preview['regions'])} regions")
print("No solver files changed; the mesh cell publishes mesh + manifest together.")


## 6. Build and validate the conformal Gmsh mesh

OCC extrudes every non-overlapping region and fragments all touching solids
so material interfaces share the same geometric surfaces.  Physical volume
groups are the solver's cell tags; exposed top/bottom faces and every
material-pair interface receive unique facet tags.

The meshing policy is explicit and visible in the diagnostics:

- first-order tetrahedra;
- Frontal-Delaunay surface meshing and Delaunay volume meshing;
- `GDS_MESH_SIZE_UM` in background material;
- a Distance/Threshold field down to `GDS_FINE_MESH_SIZE_UM` around
  conductor and source surfaces;
- deterministic seed, Gmsh optimization, and Netgen optimization.

Validation checks complete tag ownership, positive volume, analytic/mesh
volume agreement, minimum `minSICN` quality, and CAD connectivity.  The
SKY130 substrate/dielectric model must form one connected thermal domain.

The mesh and JSON are built from one fresh snapshot of the current regions.
Files are staged until all validation succeeds, and the sealed JSON is
published last. The seal verifies both mesh bytes and the exact
region/tag/material contract used by the mesher; an older on-disk JSON
cannot be silently resealed. A failed mesh build leaves the previous valid
bundle untouched. After changing the selected GDS/top cell, rerun extraction
and region construction before meshing.


In [ ]:
from mesh.gds_notebook import build_notebook_mesh_bundle

# This always snapshots CURRENT regions/process_model. It never reads an
# earlier manifest variable or JSON file, even during a partial notebook rerun.
manifest, mesh_stats, mesh_paths = build_notebook_mesh_bundle(globals())
JSON_PATH = mesh_paths["manifest"]
MSH_PATH = mesh_paths["mesh"]
QUALITY_MSH_PATH = mesh_paths["quality"]
BREP_PATH = mesh_paths["brep"]
MESH_STATS_PATH = mesh_paths["stats"]
print(
    f"Mesh: {mesh_stats['node_count']} nodes, {mesh_stats['tetrahedron_count']} tetrahedra, "
    f"{mesh_stats['shared_interface_surface_count']} shared interface surfaces"
)
for stats in mesh_stats["regions"]:
    print(
        f"tag {stats['physical_tag']:>2}: {stats['tetrahedron_count']:>6} tets, "
        f"volume={stats['mesh_volume_um3']:.6f} um^3, minSICN={stats['minimum_minSICN_quality']:.3f}"
    )
print("Solver-ready SRAM manifest:" if SRAM_PIPELINE else "Material manifest:", JSON_PATH)


## 7. Visualize the PDK geometry and the actual finite-element mesh

The first SVG shows material/source masks and an embedded-conductor 3D
cutaway.  Dielectric is rendered as a wireframe envelope so it does not hide
the interconnect, while the manifest and mesh still contain the full fill.
The second SVG uses the saved Gmsh tetrahedra: a transparent region envelope,
one true tetrahedral cut, quality distribution, realized sizes, and the
refinement policy.

The `*_material_regions_quality.msh` artifact also contains an ElementData
view named `Tetrahedron minSICN`.  With Gmsh available in your environment, open it
interactively with:

```bash
gmsh out/gds_geometry/sky130_fd_sc_hd__inv_1/sky130_fd_sc_hd__inv_1/\
  sky130_fd_sc_hd__inv_1_sky130_fd_sc_hd__inv_1_material_regions_quality.msh
```


In [ ]:
SVG_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}_geometry.svg"
svg_text = make_svg(regions, layout, GDS_PATH.name, all(region["placeholder"] for region in regions))
SVG_PATH.write_text(svg_text, encoding="utf-8")
display(SVG(svg_text))

mesh_preview = read_gmsh_mesh_for_preview(MSH_PATH)
MESH_DIAGNOSTICS_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}_mesh_diagnostics.svg"
mesh_diagnostics_svg = make_mesh_diagnostics_svg(mesh_preview, regions, mesh_stats)
MESH_DIAGNOSTICS_PATH.write_text(mesh_diagnostics_svg, encoding="utf-8")
display(SVG(mesh_diagnostics_svg))

display(HTML(region_table_html(regions, mesh_stats)))

ARTIFACTS = (
    AUDIT_PATH, JSON_PATH, BREP_PATH, MSH_PATH, QUALITY_MSH_PATH, MESH_STATS_PATH, SVG_PATH,
    MESH_DIAGNOSTICS_PATH,
)
assert all(path.is_file() and path.stat().st_size > 0 for path in ARTIFACTS)
print("Validation passed. Artifacts:")
for path in ARTIFACTS:
    print(" -", path.relative_to(ROOT))


## 8. Run Kelvin with this saved SRAM mesh

The completed MSH and material/source manifest are sealed together from the
same current-state snapshot in the mesh cell. The earlier manifest cell is only
a preview and may safely be skipped when remeshing edited region materials. Kelvin imports
them **without remeshing**, matches the eight channel polygons to verified X0–X7
instances, derives individual heat from validated transient-SPICE waveforms, and checks each integrated source.

Run from the Kelvin repository root in the DOLFINx/ngspice environment:

```bash
python cases/run_bitcell_compact.py --mesh-manifest out/sram_notebook_mesh/sram_sp_cell_material_regions.json
python cases/run_bitcell_transient.py --point read_1 --instantaneous --mesh-manifest out/sram_notebook_mesh/sram_sp_cell_material_regions.json
```

Both default to READ-1 with layout-linked wire/contact Joule heat: the steady case averages the combined SPICE event energy, while `--instantaneous` follows the transistor and resistor waveforms. The saved mesh is reused without remeshing. Both retain periodic x/y boundaries and the previous backside Robin cooling.
Use `--interconnect none` for the previous ideal-wire model. The new resistance model retains estimated bitline capacitance; it is not full foundry RC extraction. See [interconnect heating](INTERCONNECT_HEATING.md).
`--mesh-method legacy --interconnect none` selects the older bounding-box mesher. Other array/gallery
entry points are not migrated. Mesh generation alone requires no SPICE.

Set `RUN_KELVIN_SRAM=True` below to launch a steady solve in an external DOLFINx
interpreter; the notebook can keep its geometry-only kernel.
For generic cells, use `mesh.gds_import.load_tagged_msh` with
`source_power_by_tag` (watts), optional `source_device_by_tag`, or
`source_density_by_tag` (W/m³). Generic GDS does not supply device identities.

See [the pipeline guide](SRAM_NOTEBOOK_PIPELINE.md) and
[generic import interface](GDS_IMPORT.md).


In [ ]:
RUN_KELVIN_SRAM = False
if SRAM_PIPELINE:
    import subprocess
    kelvin_python = os.environ.get('KELVIN_PYTHON', sys.executable)
    kelvin_cmd = [kelvin_python, 'cases/run_bitcell_compact.py', '--mesh-manifest', str(JSON_PATH),
                  '--point', 'read_1', '--interconnect', 'layout', '--out-dir', str(OUTPUT_DIR / 'thermal_read_spice_interconnect'), '--no-renders']
    print('Saved mesh:', MSH_PATH)
    print('Steady command:', kelvin_cmd)
    if RUN_KELVIN_SRAM:
        kelvin_env = os.environ.copy()
        kelvin_env['PYTHONPATH'] = os.pathsep.join(filter(None, [
            str(ROOT), str(ROOT / '.deps/dolfinx-mpc-0.10/python'),
            kelvin_env.get('PYTHONPATH', '')]))
        kelvin_env.setdefault('UCX_TLS', 'self')
        subprocess.run(kelvin_cmd, cwd=ROOT, env=kelvin_env, check=True)
else:
    print('Generic mode: provide physical-tag powers through mesh.gds_import.load_tagged_msh.')
